# Person A — Day 5: OpenUnlearning source-helper integration

Use the same T4, FP16 and SDPA setup. This notebook calls the **unchanged pinned OpenUnlearning preprocessing, collator, probability and ROUGE helper code** on the same 20 already exposed forget-author questions, for full/retain90/SimNPO.

This is a function-level integration check, **not a full Hydra CLI run or official benchmark reproduction**. It avoids training dependencies and broad evaluation over untouched forget authors. Stock helper source and MIT license are bundled with exact hashes at commit b29929067732c40ee4ca58d5cc1d3c11fcf37de1.

We also run clearly labeled diagnostics: stock attention masks versus all-ones masks on unpadded batch-size-one inputs; an independent FP32 reference probability calculation; and comparison with previously saved ordinary generations. No quantization/training is involved. Source code is inspected, not silently patched.

There are 120 short generation calls (20 items × 2 mask conditions × 3 models), plus reference scoring. Only one model occupies the GPU at a time. Locally checked only; GPU execution is pending.


In [ ]:
print("Execution test: Python is running", flush=True)


## Setup

On a fresh runtime, run this cell. On an existing successful runtime, the three added helper dependencies (`omegaconf`, `rouge-score`, `scipy`) must still be available. Restart the kernel if prompted, then skip installation and continue from the environment cell. Do not install the full upstream training stack or change Torch silently.


In [ ]:
%pip install "transformers==5.5.4" "huggingface-hub==1.7.2" "datasets==3.0.1" "accelerate==1.13.0" "omegaconf==2.3.0" "rouge-score==0.1.2" scipy


In [ ]:
import copy,csv,gc,hashlib,importlib.util,json,math,subprocess,sys,time,zipfile
from datetime import datetime,timezone
from pathlib import Path
import numpy as np
import torch,transformers,datasets,accelerate,huggingface_hub,scipy,omegaconf,yaml
from omegaconf import OmegaConf
from transformers import AutoModelForCausalLM,AutoTokenizer,GenerationConfig
assert torch.cuda.is_available(),"Enable the T4 GPU and reconnect."
assert transformers.__version__=="5.5.4","Restart after installing the recorded version."
ROOT=Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUT=ROOT/"week1_artifacts"/datetime.now(timezone.utc).strftime("day5_%Y%m%dT%H%M%S%fZ")
OUT.mkdir(parents=True,exist_ok=False)
ENV=dict(python=sys.version,torch=torch.__version__,transformers=transformers.__version__,datasets=datasets.__version__,
 accelerate=accelerate.__version__,huggingface_hub=huggingface_hub.__version__,cuda_build=torch.version.cuda,
 gpu=torch.cuda.get_device_name(0),dtype="float16",attention="sdpa",scipy=scipy.__version__,omegaconf=omegaconf.__version__)
(OUT/"environment.json").write_text(json.dumps(ENV,indent=2))
(OUT/"pip-freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True,timeout=60))
print(json.dumps(ENV,indent=2));print("Artifacts:",OUT)


## Upload the pinned source/input ZIP

Colab: leave the path blank and select only `day5-source-parity-input-bundle.zip`. Kaggle: upload/add it privately and enter the mounted path. The ZIP includes source code, exact fixtures, previous generations and provenance, not model weights or credentials. The source manifest states the scope/adaptations explicitly.


In [ ]:
INPUT_BUNDLE=""
EXPECTED_BUNDLE_SHA256="b6dead9080b27077f0ffa55a06346bcde36879e9997bc1dc9d8a980b67a10e36"
if not INPUT_BUNDLE:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Set INPUT_BUNDLE to the uploaded ZIP's mounted path in Kaggle.")
    uploaded=files.upload();names=[n for n in uploaded if n.endswith(".zip")]
    assert len(names)==1,"Upload exactly one input ZIP."
    INPUT_BUNDLE=names[0]
bundle=Path(INPUT_BUNDLE)
assert hashlib.sha256(bundle.read_bytes()).hexdigest()==EXPECTED_BUNDLE_SHA256,"Wrong/modified bundle."
with zipfile.ZipFile(bundle) as z:
    source_manifest=json.loads(z.read("source-manifest.json"))
    fixtures=json.loads(z.read("references/fixtures.json"))
    common_dict=json.loads(z.read("references/effective-generation-config.json"))
    reference_env=json.loads(z.read("reference-environment.json"))
    provenance=json.loads(z.read("provenance.json"))
    template_bytes=z.read("references/chat-template.txt")
    previous={role:{r["item_id"]:r for r in map(json.loads,z.read(f"references/{role}-outputs.jsonl").decode().splitlines())}
              for role in ("full","retain90","simnpo_candidate")}
    for name,digest in source_manifest["files"].items():
        relative=Path(name)
        assert not relative.is_absolute() and ".." not in relative.parts
        payload=z.read("upstream/"+name)
        assert hashlib.sha256(payload).hexdigest()==digest
        target=OUT/"upstream"/relative;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(payload)
assert len(fixtures)==20
for key in ("torch","transformers","datasets","accelerate","huggingface_hub","cuda_build","gpu","dtype","attention"):
    assert ENV[key]==reference_env[key],f"Environment drift for {key}; review before proceeding."
for role,rows in previous.items():
    assert set(rows)=={f["item_id"] for f in fixtures}
    for f in fixtures:assert rows[f["item_id"]]["prompt_token_ids"]==f["prompt_token_ids"]
(OUT/"source-manifest.json").write_text(json.dumps(source_manifest,indent=2))
MODELS={role:provenance["models"][role] for role in ("full","retain90","simnpo_candidate")}
manifest=dict(kind="source_helper_integration_development",phase="development",run_id=OUT.name,
 models=MODELS,tokenizer=MODELS["full"],input_bundle_sha256=EXPECTED_BUNDLE_SHA256,
 source_commit=source_manifest["commit"],scope=source_manifest["execution_scope"],
 item_ids=[f["item_id"] for f in fixtures],status="prepared",stock_functions_unmodified=True,
 probability_definition="exp(-mean target-token cross entropy), including upstream labeled EOS convention",
 limitations=source_manifest["adaptations"])
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
print("Verified source hashes and 20 reused development items.")


## Load actual upstream helpers and trace tokenization/masks

Importing `src/eval.py` would pull in training/evaluator registries. Instead, this harness imports the real data package and loads `src/evals/metrics/utils.py` directly from its file. The helper bodies are unchanged; no registry behavior or full config composition is claimed.

The upstream collator masks every ID equal to the padding token. With pad=EOS, genuine chat markers can be affected even without padding. We save the masks and token positions, and compare an all-ones mask on the same unpadded sequences. This characterizes an implementation difference; do not assume any magnitude or call it a bug before inspecting the evidence.


In [ ]:
UPSTREAM=OUT/"upstream"
SRC=UPSTREAM/"src"
if "data" in sys.modules:
    raise RuntimeError("A data module is already loaded; restart in a fresh kernel to avoid package-name conflicts.")
sys.path.insert(0,str(SRC))
from data.utils import preprocess_chat_instance,IGNORE_INDEX
from data.collators import DataCollatorForSupervisedDataset
spec=importlib.util.spec_from_file_location("kv_pinned_stock_metric_utils",SRC/"evals/metrics/utils.py")
stock=importlib.util.module_from_spec(spec);spec.loader.exec_module(stock)
model_yaml=yaml.safe_load((UPSTREAM/"configs/model/Llama-3.2-1B-Instruct.yaml").read_text())
template_args=model_yaml["template_args"]
assert template_args["date_string"]=="10 Apr 2025"
tok=AutoTokenizer.from_pretrained(MODELS["full"]["id"],revision=MODELS["full"]["revision"])
if tok.pad_token_id is None:tok.pad_token=tok.eos_token
assert tok.chat_template.encode()==template_bytes
assert tok.pad_token_id==tok.eos_token_id==common_dict["pad_token_id"]
prob_collator=DataCollatorForSupervisedDataset(tok,padding_side="right")
gen_collator=DataCollatorForSupervisedDataset(tok,padding_side="left")
prepared=[];trace=[]
for fixture in fixtures:
    kwargs=dict(tokenizer=tok,template_config=template_args,prompt_msgs=[fixture["question"]],
                response_msgs=[fixture["gold_answer"]],max_length=512)
    probability_item=preprocess_chat_instance(**kwargs,predict_with_generate=False)
    generation_item=preprocess_chat_instance(**kwargs,predict_with_generate=True)
    assert generation_item["input_ids"].tolist()==fixture["prompt_token_ids"],"Stock prompt differs from prior baseline."
    prompt_len=len(fixture["prompt_token_ids"])
    assert probability_item["input_ids"][:prompt_len].tolist()==fixture["prompt_token_ids"],"Answer boundary/tokenization differs."
    assert bool((probability_item["labels"][:prompt_len]==IGNORE_INDEX).all())
    prob_batch=prob_collator([probability_item]);gen_batch=gen_collator([generation_item])
    assert prob_batch["input_ids"].shape[0]==gen_batch["input_ids"].shape[0]==1
    labels=prob_batch["labels"]
    assert int((labels[:,1:]!=IGNORE_INDEX).sum())==int((labels!=IGNORE_INDEX).sum())>0
    row=dict(item_id=fixture["item_id"],prompt_tokens=prompt_len,
        scoring_input_ids=prob_batch["input_ids"][0].tolist(),scoring_labels=labels[0].tolist(),
        stock_scoring_mask=prob_batch["attention_mask"][0].int().tolist(),
        stock_generation_mask=gen_batch["attention_mask"][0].int().tolist(),
        zero_positions_in_unpadded_prompt=[i for i,v in enumerate(gen_batch["attention_mask"][0].tolist()) if not v],
        target_token_count=int((labels!=IGNORE_INDEX).sum()),last_labeled_token_id=int(labels[0,-1]))
    trace.append(row);prepared.append((fixture,prob_batch,gen_batch))
(OUT/"token-and-mask-trace.json").write_text(json.dumps(trace,indent=2))
print("pad/eos ID:",tok.pad_token_id)
print("First prompt's stock zero-mask positions:",trace[0]["zero_positions_in_unpadded_prompt"])
print("Scored answer-token count (upstream EOS convention):",trace[0]["target_token_count"])


## Independent score calculation and generation capture

The independent FP32 calculation is a diagnostic on the same tokens and masks. Upstream uses its own cross-entropy implementation/dtype. Save differences rather than inventing a tolerance after looking at outcomes.

`prob` is a geometric mean of target-token probabilities, not sequence probability, factual correctness, or proof of latent knowledge. Both paths use the upstream answer/EOS boundaries. No cache quantization is active.


In [ ]:
def reference_probability(model,batch):
    inputs={k:v.to(model.device) for k,v in batch.items()}
    with torch.inference_mode():
        output=model(**inputs)
    labels=inputs["labels"][:,1:]
    logits=output.logits[:,:-1,:].float()
    valid=labels!=IGNORE_INDEX
    assert int(valid.sum())>0
    safe_labels=labels.masked_fill(~valid,0)
    nll=-logits.log_softmax(-1).gather(-1,safe_labels.unsqueeze(-1)).squeeze(-1)
    mean_nll=float(nll[valid].mean())
    return dict(mean_nll=mean_nll,prob=math.exp(-mean_nll),target_tokens=int(valid.sum()))

def clone_batch(batch,all_ones=False):
    result={k:v.clone() for k,v in batch.items()}
    if all_ones:result["attention_mask"]=torch.ones_like(result["attention_mask"])
    return result

common=GenerationConfig.from_dict(common_dict)
assert common.do_sample is False and common.cache_implementation=="dynamic"
# All remaining defaults are shared by assigning common to each loaded model.
keys=("do_sample","num_beams","max_new_tokens","use_cache","cache_implementation", "temperature","top_k","top_p","eos_token_id")
generation_args={key:getattr(common,key) for key in keys}
(OUT/"helper-generation-overrides.json").write_text(json.dumps(generation_args,indent=2))

def captured_similarity(model,batch):
    captured={}
    original_generate=model.generate
    def recording_generate(*args,**kwargs):
        result=original_generate(*args,**kwargs)
        assert isinstance(result,torch.Tensor),"Expected token tensor from shared generation config."
        captured["sequences"]=result.detach().cpu().tolist()
        return result
    model.generate=recording_generate
    try:
        result=stock.eval_text_similarity(model,tok,clone_batch(batch),OmegaConf.create(generation_args))[0]
    finally:
        model.generate=original_generate
    prompt_len=batch["input_ids"].shape[1]
    generated_ids=captured["sequences"][0][prompt_len:]
    eos=common.eos_token_id if isinstance(common.eos_token_id,list) else [common.eos_token_id]
    result.update(generated_token_ids=generated_ids,generated_token_count=len(generated_ids),
        ended_with_eos=bool(generated_ids and generated_ids[-1] in eos),
        reached_token_budget=len(generated_ids)>=common.max_new_tokens)
    return result


## Run the integration comparison

Two declared mask conditions use the same unpadded inputs. We never change the source collator to make a comparison pass. An independent FP32 reference is computed only for the stock-mask probability condition. Stock-helper ROUGE generation uses batch size one and 64 tokens; upstream default config uses a different budget, so this is not full published-score reproduction.

Every item is saved immediately. A genuine runtime failure stops the run and preserves diagnostics. Different text between runs is an observation, not automatically a cache effect. The all-ones helper generations can also be compared with earlier saved text to check the new harness.


In [ ]:
for name in ("model","loaded","inputs","output","generated","ids","mask"):
    globals().pop(name,None)
gc.collect();torch.cuda.empty_cache()
ALL_RECORDS=[];SUMMARIES={}
for role,checkpoint in MODELS.items():
    path=OUT/f"{role}-source-helper-results.jsonl"
    if path.exists():raise RuntimeError("Preserve existing results; use a new documented run for retries.")
    model=None
    try:
        manifest["status"]=f"running_{role}"
        (OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
        print("Loading",role,flush=True);started=time.perf_counter()
        model=AutoModelForCausalLM.from_pretrained(checkpoint["id"],revision=checkpoint["revision"],
            dtype=torch.float16,attn_implementation="sdpa").to("cuda").eval()
        model.generation_config=copy.deepcopy(common)
        torch.cuda.synchronize();load_seconds=time.perf_counter()-started
        role_records=[]
        with path.open("x") as handle:
            for number,(fixture,prob_batch,gen_batch) in enumerate(prepared,1):
                record=dict(item_id=fixture["item_id"],model_role=role,checkpoint=checkpoint,
                    kind="source_helper_integration",phase="development",status="ok",error=None)
                try:
                    started=time.perf_counter()
                    p_stock=stock.evaluate_probability(model,clone_batch(prob_batch))[0]
                    p_ones=stock.evaluate_probability(model,clone_batch(prob_batch,all_ones=True))[0]
                    ref=reference_probability(model,clone_batch(prob_batch))
                    g_stock=captured_similarity(model,gen_batch)
                    g_ones=captured_similarity(model,clone_batch(gen_batch,all_ones=True))
                    assert all(math.isfinite(float(x)) for x in (p_stock["prob"],p_stock["avg_loss"],p_ones["prob"],p_ones["avg_loss"],ref["mean_nll"],ref["prob"]))
                    record.update(stock_mask_probability=p_stock,all_ones_mask_probability=p_ones,
                        independent_fp32_stock_mask=ref,
                        stock_minus_fp32_mean_nll=p_stock["avg_loss"]-ref["mean_nll"],
                        stock_mask_generation=g_stock,all_ones_mask_generation=g_ones,
                        same_generation_between_masks=g_stock["generation"]==g_ones["generation"],
                        all_ones_matches_previous_text=g_ones["generation"].strip()==previous[role][fixture["item_id"]]["generated_text"].strip(),
                        original_gold_answer=fixture["gold_answer"],question=fixture["question"],
                        helper_ground_truth_matches_original=g_stock["ground_truth"].strip()==fixture["gold_answer"].strip(),
                        elapsed_seconds=time.perf_counter()-started)
                except Exception as exc:
                    record.update(status="error",error=dict(type=type(exc).__name__,message=str(exc)))
                handle.write(json.dumps(record,ensure_ascii=False,allow_nan=False)+"\n");handle.flush()
                ALL_RECORDS.append(record);role_records.append(record)
                print(role,number,"/20",record["status"],flush=True)
                if record["status"]!="ok":raise RuntimeError("Saved item error; stop and diagnose before proceeding.")
        SUMMARIES[role]=dict(items=20,load_seconds=load_seconds,
            stock_mask_mean_prob=float(np.mean([r["stock_mask_probability"]["prob"] for r in role_records])),
            all_ones_mean_prob=float(np.mean([r["all_ones_mask_probability"]["prob"] for r in role_records])),
            stock_mask_mean_rougeL_recall=float(np.mean([r["stock_mask_generation"]["rougeL_recall"] for r in role_records])),
            all_ones_mean_rougeL_recall=float(np.mean([r["all_ones_mask_generation"]["rougeL_recall"] for r in role_records])),
            changed_generation_count=sum(not r["same_generation_between_masks"] for r in role_records),
            all_ones_previous_text_matches=sum(r["all_ones_matches_previous_text"] for r in role_records),
            helper_ground_truth_matches=sum(r["helper_ground_truth_matches_original"] for r in role_records),
            max_abs_stock_fp32_mean_nll_delta=max(abs(r["stock_minus_fp32_mean_nll"]) for r in role_records))
    except Exception as exc:
        manifest["status"]="failed_partial_results_preserved";manifest["error"]=f"{type(exc).__name__}: {exc}"
        (OUT/"manifest.json").write_text(json.dumps(manifest,indent=2));raise
    finally:
        if model is not None:del model
        gc.collect();torch.cuda.empty_cache()
        (OUT/"summary.json").write_text(json.dumps(SUMMARIES,indent=2))
assert len(ALL_RECORDS)==60
manifest["status"]="integration_complete_review_pending";manifest["completed_items"]=60
# Confirm the helper source files remain unchanged after execution.
for name,digest in source_manifest["files"].items():
    assert hashlib.sha256((UPSTREAM/name).read_bytes()).hexdigest()==digest
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
print(json.dumps(SUMMARIES,indent=2))


## Save the evidence and return it for interpretation

The output ZIP contains source hashes/files, token/mask traces, per-item scores and generations, and per-model summaries. Keep all differences visible. A source-helper discrepancy is not yet a published evaluator bug or a quantization result.

Return the ZIP. We will inspect token boundaries, reference reconstruction, numerical differences, attention masks, and agreement with your prior generations before any next cache experiment. The full CLI/configuration integration, TruthRatio/forget-quality and official utility suite remain separate unfinished steps. No new held-out forget author has been evaluated.


In [ ]:
import shutil
bundle_path=shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print("DOWNLOAD:",bundle_path)
try:
    from google.colab import files
    files.download(bundle_path)
except ImportError:
    print("Download the ZIP from Kaggle's file browser.")
